# Hearing a figure come apart

A cloud of short tones. Hidden inside it, seven of them keep returning at the same
pitches, and you hear those seven as one thing because they start together.

What if they stop starting together? Below, each tone of the figure is delayed behind
the one under it, from 0 ms up to 50 ms, and you can listen for where it stops holding.

Teki (2013) and O'Sullivan (2015) slid the figure around in frequency and it survived.
Sliding it in time is the case where the textbook says it should break.

Headphones, both ears. Runtime > Run all. Setup takes about 20 seconds.

In [ ]:
# @title Setup: fetch the code and warm up (click the play button, about 20 s)
import os, subprocess, sys

REPO = "https://github.com/MeysamAmirsardari/Rate_RNN.git"
if not os.path.isdir("Rate_RNN"):
    # sparse clone: the repo is 5 GB, the stimulus code is 60 kB
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "--filter=blob:none", "--sparse", REPO], check=True)
    subprocess.run(["git", "-C", "Rate_RNN", "sparse-checkout", "set", "--no-cone",
                    "/audios/*.py", "/audios/sfg/*.py", "/audios/sfg_task/*.py"],
                   check=True)
sys.path.insert(0, os.path.abspath("Rate_RNN"))

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram as _spec
from IPython.display import Audio, HTML, display

from audios.sfg_task.config import Design
from audios.sfg_task.stimulus import make_pool, trial
from audios.sfg_task.plot import raster_ax, RED

D = Design()          # every parameter of the experiment lives in this one object
D.validate()
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

_pools = {}
def pool_for(d):
    """Channel grid for a design. Cached; it never changes within a run."""
    key = (d.f_lo, d.f_hi, d.grid_st, d.phon, d.min_sep_erb)
    if key not in _pools:
        _pools[key] = make_pool(d)
    return _pools[key]

def build(step_ms, seed=0, variant="rise", d=None):
    """One trial: the interval with the figure, and the matched one without."""
    d = d or D
    return trial(d, pool_for(d), step_ms=step_ms, seed=seed,
                 variant=variant, rove=False)

def player(clip, gain, fs=None):
    """An audio player at a gain you choose, so levels stay comparable."""
    return Audio(clip * gain, rate=fs or D.fs, normalize=False)._repr_html_()

def gain_for(*clips):
    return 0.89 / max(float(np.abs(c).max()) for c in clips)

def players(clips, labels, fs=None):
    g = gain_for(*clips)
    return HTML("".join(
        f"<div style='margin:2px 0 12px'><b>{l}</b><br>{player(c, g, fs)}</div>"
        for c, l in zip(clips, labels)))

def score(schs, titles, d=None, seconds=3.0):
    """One dot per tone, the figure's in red. Only the first few seconds:
    six seconds of a thousand tones in one figure is a smear."""
    d = d or D
    fig, ax = plt.subplots(len(schs), 1, figsize=(11, 2.15 * len(schs)),
                           sharex=True, sharey=True, squeeze=False,
                           constrained_layout=True)
    for a, sch, t in zip(ax[:, 0], schs, titles):
        raster_ax(d, pool_for(d), sch, a, seconds, bg="0.72", ms=3.2)
        a.set_title(t, fontsize=10, loc="left")
        a.set_ylabel(f"st re {d.f_lo:.0f} Hz", fontsize=8)
    ax[-1, 0].set_xlabel(f"time (s), first {seconds:g} of {d.interval_s:g}")
    plt.show()

def spectro(y, title="", fs=None, top_db=55):
    """The spectrogram."""
    fs = fs or D.fs
    f, t, S = _spec(y, fs, nperseg=1024, noverlap=896)
    S = 10 * np.log10(S + 1e-20)
    fig, ax = plt.subplots(figsize=(11, 3), constrained_layout=True)
    ax.pcolormesh(t, f, S - S.max(), vmin=-top_db, vmax=0,
                  cmap="magma", shading="auto")
    ax.set_yscale("log"); ax.set_ylim(D.f_lo * .9, D.f_hi * 1.1)
    ax.set_ylabel("frequency (Hz)"); ax.set_xlabel("time (s)"); ax.set_title(title)
    plt.show()

print("ready\n")
print(D.summary())

---
## 1. The cloud

Five tones sounding at any moment, 50 ms each, drawn from 117 pitches over five
octaves, all at the same level. No two ever sound at once inside the same critical
band, so nothing beats against anything and no pitch stands out from its neighbours.

The plot is a score: one dot per tone, time across, pitch up. Red dots are the ones
that would be the figure. Here they land somewhere new every time, so there is nothing
to hold onto. It should sound like rain on a window.

In [ ]:
_, cloud = build(step_ms=20, seed=3)          # the figure-absent interval
display(players([cloud["y"]], ["the cloud, nothing hiding in it"]))
score([cloud], ["no figure: every group lands somewhere new"])
spectro(cloud["y"], "the same six seconds, as a spectrogram")

---
## 2. The figure

Seven tones, always the same seven pitches, all starting at once, coming back about
five times a second across the six seconds, at an irregular rhythm.

Hear it alone first, then the same seven tones buried in the cloud.

In [ ]:
figure, nofigure = build(step_ms=0, seed=11)
g = gain_for(figure["y"], nofigure["y"], figure["y_fig"])
display(HTML(
    "<div style='margin:2px 0 12px'><b>the figure alone</b><br>"
    + player(figure["y_fig"], g) + "</div>"
    "<div style='margin:2px 0 12px'><b>the same figure, in the cloud</b><br>"
    + player(figure["y"], g) + "</div>"
    "<div style='margin:2px 0 12px'><b>the matched interval with no figure</b><br>"
    + player(nofigure["y"], g) + "</div>"))
score([figure, nofigure], ["figure present: the red rows repeat",
                           "figure absent: same rhythm, new pitches each time"])

---
## 3. Smearing it

Each row delays every tone of the figure a bit further behind the one below it. The red
dots tip from vertical to diagonal. Nothing else changes: same seven pitches, same
number of repetitions, same number of tones in the cloud, same loudness.

One thing does change, and it is worth knowing. The figure repeats every 200 ms, and at
50 ms per tone one repetition lasts 350 ms, so at the wide delays the repetitions run
into each other and the figure becomes a continuous run rather than a series of events.
It sounds a figure tone 23% of the time at 0 ms and 93% of the time at 50 ms.

Work down the list. Where does it stop being something you notice?

In [ ]:
rows, cells_html = [], []
for step in D.steps_ms:
    fig_iv, _ = build(step_ms=step, seed=21)
    g = gain_for(fig_iv["y"], fig_iv["y_fig"])
    label = "all at once, a chord" if step == 0 else \
            f"{D.extent_ms(step):.0f} ms from first tone to last"
    cells_html.append(
        f"<tr><td style='padding:6px 14px;white-space:nowrap'>"
        f"<b style='font-size:15px'>{step:g} ms</b><br>"
        f"<span style='color:#888;font-size:11px'>{label}</span></td>"
        f"<td style='padding:6px'>{player(fig_iv['y'], g)}</td>"
        f"<td style='padding:6px'>{player(fig_iv['y_fig'], g)}</td></tr>")
    rows.append(fig_iv)

display(HTML(
    "<table style='border-collapse:collapse'><tr>"
    "<th style='text-align:left;padding:4px 14px'>delay per tone</th>"
    "<th style='text-align:left;padding:4px'>in the cloud</th>"
    "<th style='text-align:left;padding:4px'>the figure alone</th></tr>"
    + "".join(cells_html) + "</table>"))

fig, ax = plt.subplots(len(rows), 1, figsize=(11, 1.25 * len(rows)),
                       sharex=True, sharey=True, constrained_layout=True)
for a, sch, step in zip(ax, rows, D.steps_ms):
    raster_ax(D, pool_for(D), sch, a, seconds=3.0, bg="0.72", ms=3.2)
    a.set_ylabel(f"{step:g} ms", rotation=0, ha="right", va="center", fontsize=11)
ax[-1].set_xlabel("time (s)")
plt.show()

---
## 4. Which one had the figure?

Two intervals, one of them has a figure. They are matched on rhythm, on tone count and
on loudness to a hundredth of a decibel.

Listen to A, then B, decide, then open the answer. Re-run the cell for a new pair; the
delay is drawn at random each time.

In [ ]:
import random

step = random.choice(D.steps_ms)
present, absent = build(step_ms=step, seed=random.randrange(10 ** 6))
first = random.choice([0, 1])
ivs = [present, absent] if first == 0 else [absent, present]
g = gain_for(ivs[0]["y"], ivs[1]["y"], present["y_fig"])

answer = "AB"[first]
shape = "a chord, all at once" if step == 0 else (
    "a staircase %.0f ms long" % D.extent_ms(step))

display(HTML(
    "<div style='margin:2px 0 12px'><b>Interval A</b><br>"
    + player(ivs[0]["y"], g) + "</div>"
    "<div style='margin:2px 0 12px'><b>Interval B</b><br>"
    + player(ivs[1]["y"], g) + "</div>"
    "<details style='margin-top:10px;padding:12px 16px;background:#f4f4f6;"
    "border-radius:8px;max-width:560px'>"
    "<summary style='cursor:pointer;font-weight:600'>Reveal</summary>"
    "<p style='margin:10px 0 4px'>The figure was in "
    "<b style='font-size:16px'>interval " + answer + "</b>.</p>"
    "<p style='margin:4px 0;color:#555'>Its tones were "
    + ("%g" % step) + " ms apart, " + shape + ".<br>"
    "Here it is alone; then play interval " + answer + " again.</p>"
    + player(present["y_fig"], g)
    + "<p style='margin:10px 0 0;color:#888;font-size:12px'>"
    "Re-run the cell for another pair.</p></details>"))
score(ivs, ["Interval A", "Interval B"])

---
## 5. Knobs

Everything lives in one object, so you can turn any of it and hear the result. Move the
sliders and press Run Interact.

| knob | what it does |
|---|---|
| delay per tone | 0 ms is a chord, 50 ms is a staircase. The one variable |
| figure tones | fewer tones is a harder task |
| rate | how often the figure comes back, per second |
| cloud density | tones sounding at once. Thin it out and the figure jumps out |
| tone length | short tones are clicky, long ones smear together |
| direction | a staircase going up, or coming down |

Some combinations do not fit in the interval, and it will say so.

In [ ]:
import ipywidgets as W

@W.interact_manual(
    step_ms=W.SelectionSlider(options=[0, 5, 10, 20, 30, 40, 50], value=20,
                              description="delay/tone"),
    coherence=W.IntSlider(value=7, min=2, max=12, description="figure tones"),
    rate_hz=W.SelectionSlider(options=[1, 2, 3, 4, 5, 6], value=5,
                              description="rate (Hz)"),
    bg_sounding=W.IntSlider(value=5, min=3, max=14, description="cloud density"),
    tone_ms=W.SelectionSlider(options=[25, 50, 75], value=50, description="tone (ms)"),
    order=W.Dropdown(options=["rise", "fall"], value="rise", description="direction"),
    seed=W.IntSlider(value=0, min=0, max=99, description="seed"))
def playground(step_ms, coherence, rate_hz, bg_sounding, tone_ms, order, seed):
    d = D.replace(
        coherence=coherence, rate_hz=float(rate_hz), bg_sounding=bg_sounding,
        tone_ms=float(tone_ms), order=order,
        steps_ms=tuple(sorted({0.0, float(step_ms), 50.0})))
    try:
        d.validate()
        present, absent = build(step_ms=float(step_ms), seed=seed, d=d)
    except ValueError as e:
        return print(f"that combination does not fit: {e}")
    g = gain_for(present["y"], absent["y"], present["y_fig"])
    display(HTML(
        f"<b>{coherence} tones, {step_ms:g} ms apart "
        f"({d.extent_ms(step_ms):.0f} ms per repetition), {d.events}x at "
        f"{rate_hz} Hz, {bg_sounding} tones of cloud</b>"
        f"<div style='margin:8px 0'>figure alone{player(present['y_fig'], g, d.fs)}</div>"
        f"<div style='margin:8px 0'>in the cloud{player(present['y'], g, d.fs)}</div>"
        f"<div style='margin:8px 0'>matched, no figure{player(absent['y'], g, d.fs)}</div>"))
    score([present, absent], ["figure", "no figure"], d=d)

---
## 6. Controls

Three questions a referee will ask, and the stimulus for each. All at a 20 ms delay.

**shuffled.** The same seven delays, out of order, so it is no longer a rising sweep.
Is the effect about asynchrony, or about the tune it plays?

**unfrozen.** The same seven pitches arriving in the same window, but the delays are
redrawn every repetition, so there is no fixed pattern to learn. Does the pattern
matter, or only that the pitches come back?

**scattered.** The same seven pitches at the same rate, never grouped into repetitions.
Its long-term spectrum is identical to the figure's. This is the one that separates
temporal coherence from the figure just being spectrally prominent.

In [ ]:
CONTROLS = [("rise",    "the figure itself, 20 ms staircase"),
            ("perm",    "shuffled: same delays, wrong order"),
            ("redraw",  "unfrozen: delays redrawn every repetition"),
            ("scatter", "scattered: same pitches, never grouped")]

built = [build(step_ms=20, seed=5, variant=v)[0] for v, _ in CONTROLS]
g = gain_for(*[b["y"] for b in built])
display(HTML("".join(
    f"<div style='margin:2px 0 12px'><b>{name}</b><br>{player(b['y'], g)}</div>"
    for (v, name), b in zip(CONTROLS, built))))
score(built, [n for _, n in CONTROLS])

---
## 7. What is held constant

The design rests on the two intervals differing in coherence and in nothing else, so
that gets measured on freshly built stimuli every time rather than asserted. This is
the table for the supplement.

Rows read figure / no figure. The last three lines are the differences that would be
cues if they were not zero, and the bottom line is what the same measurement returns
when there is nothing to find.

About 30 seconds.

In [ ]:
from audios.sfg_task.verify import verify, table

res = [verify(D, s, n=6) for s in D.steps_ms]
print(table(res))

fig, ax = plt.subplots(1, len(res), figsize=(2.1 * len(res), 2.6),
                       sharey=True, constrained_layout=True)
for a, r in zip(ax, res):
    t = np.linspace(-150, 500, r["epoch"][0].size)
    a.plot(t, r["epoch"][0], color=RED, lw=1.1)
    a.plot(t, r["epoch"][1], color="k", lw=1.1, ls="--")
    a.axvline(0, color="0.75", lw=.8); a.set_title(f"{r['step_ms']:g} ms", fontsize=9)
    a.set_xlabel("ms re repetition")
    for s in ("top", "right"): a.spines[s].set_visible(False)
ax[0].set_ylabel("loudness (dB)")
plt.suptitle("loudness around each repetition. red is figure, black is no figure, "
             "on a 0.1 dB axis", fontsize=9)
plt.show()

---
## Running it properly

140 trials, about 35 minutes, two intervals a trial, seven delays, with a practice
block, breaks and a level calibration. It comes out as a psychometric curve whose one
number is the delay at which the figure stops binding.

```bash
git clone https://github.com/MeysamAmirsardari/Rate_RNN.git
cd Rate_RNN
python -m audios.sfg_task check          # the measurements from section 7
python -m audios.sfg_task calibrate      # set 65 dB SPL once
python -m audios.sfg_task run  S01       # the experiment, resumes if interrupted
python -m audios.sfg_task analyse S01    # d', the fit, the threshold
```

The rationale, the full control battery and a list of what is not controlled are in
[`audios/sfg_task/README.md`](https://github.com/MeysamAmirsardari/Rate_RNN/blob/main/audios/sfg_task/README.md).